### Importing libs

In [1]:
import $ivy.`org.apache.spark:spark-core_2.12:3.5.3`
import $ivy.`org.apache.spark:spark-sql_2.12:3.5.3`
import $ivy.`org.postgresql:postgresql:42.7.3` // this module needs to import
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame

import $ivy.$                                       

import $ivy.$                                      

import $ivy.$                                  // this module needs to import

import org.apache.spark.sql.SparkSession

import org.apache.spark.sql.functions._

import org.apache.spark.sql.DataFrame

### defining spark

In [7]:
val spark = SparkSession.builder().master("local[*]").appName("postgreConnector").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._


spark: SparkSession = org.apache.spark.sql.SparkSession@b10a905
import spark.implicits._


### Define JDBC connection properties

In [8]:
val url = "jdbc:postgresql://localhost:5432/postgres"
val postgresConf = new java.util.Properties()
postgresConf.setProperty("user", "postgres")
postgresConf.setProperty("password", "postgres")
postgresConf.setProperty("driver", "org.postgresql.Driver")

url: String = "jdbc:postgresql://localhost:5432/postgres"
postgresConf: java.util.Properties = {password=postgres, driver=org.postgresql.Driver, user=postgres}
res7_2: Object = null
res7_3: Object = null
res7_4: Object = null

### Access table from postgres

In [14]:
val df = spark.read.jdbc(url, "test.products", postgresConf)
df.show(truncate=false)

+----------+------------+-----------------------+-----------+
|product_id|product_name|price                  |category   |
+----------+------------+-----------------------+-----------+
|1         |Laptop      |1200.000000000000000000|Electronics|
|2         |Mouse       |25.000000000000000000  |Electronics|
|3         |Keyboard    |45.000000000000000000  |Electronics|
|4         |Monitor     |200.000000000000000000 |Electronics|
|5         |Desk Chair  |150.000000000000000000 |Furniture  |
|6         |Desk Lamp   |30.000000000000000000  |Furniture  |
|7         |Notebook    |5.000000000000000000   |Stationery |
|8         |Pen Set     |12.000000000000000000  |Stationery |
|9         |Charger     |200.000000000000000000 |Electronics|
|10        |Cell        |200.000000000000000000 |Electronics|
+----------+------------+-----------------------+-----------+



df: DataFrame = [product_id: int, product_name: string ... 2 more fields]

### Running query

In [21]:
spark.read.jdbc(url, "(select * from test.products where price>100) as query", postgresConf).show(false)

+----------+------------+-----------------------+-----------+
|product_id|product_name|price                  |category   |
+----------+------------+-----------------------+-----------+
|1         |Laptop      |1200.000000000000000000|Electronics|
|4         |Monitor     |200.000000000000000000 |Electronics|
|5         |Desk Chair  |150.000000000000000000 |Furniture  |
|9         |Charger     |200.000000000000000000 |Electronics|
|10        |Cell        |200.000000000000000000 |Electronics|
+----------+------------+-----------------------+-----------+



### another way of connecting

In [17]:
spark.read.format("jdbc")
    .option("user", "postgres")
    .option("password", "postgres")
    .option("url", url)
    .option("driver", "org.postgresql.Driver")
    .option("query", "select * from test.products where price>100")
    .load().show()

+----------+------------+--------------------+-----------+
|product_id|product_name|               price|   category|
+----------+------------+--------------------+-----------+
|         1|      Laptop|1200.000000000000...|Electronics|
|         4|     Monitor|200.0000000000000...|Electronics|
|         5|  Desk Chair|150.0000000000000...|  Furniture|
|         9|     Charger|200.0000000000000...|Electronics|
|        10|        Cell|200.0000000000000...|Electronics|
+----------+------------+--------------------+-----------+

